In [ ]:
# SONDEWNPN
# Balloon-borne sounding system (BBSS): Vaisala-processed winds, press., temp, &RH
# Data Level: b1
# Site: Houston, TX; Tracking Aerosol Convection interactions ExpeRiment (HOU)
# Facility: Houston, TX; AMF1 (main site for TRACER) (M1)
# Category: Atmospheric Profiling
# Source Instrument/Data: Balloon-Borne Sounding System (sonde)
# Date Range: 2021-09-30 to 2021-11-08

In [ ]:
#Needed Libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import os

#Specific Libraries
from datetime import datetime
from metpy.plots import SkewT
from metpy.units import units
from metpy.io import parse_metar_file
import metpy.calc as mpcalc

In [ ]:
data_location='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/CodeFiles/1_Initial_Figures/Figures/Campaign_Data/dropsonde_data'
count_nc=0
nc_filenames = []
for i, file_name in enumerate(os.listdir(data_location), start=1):
    if file_name.endswith(".cdf"): 
        count_nc+=1
        nc_filenames.append(file_name) 
# print(f'num files: {len(nc_filenames)}') 

#look for all soundings at specific date
date='20220622'
print('Times to choose from on specified date: ',[t for t in nc_filenames if date in t])

hour = '11'
datehour = f"{date}.{hour}"
which_file=nc_filenames.index([t for t in nc_filenames if datehour in t][0])
print("Specified datehour filename: ",nc_filenames[int(which_file)])

data=xr.open_dataset(data_location+'/'+nc_filenames[int(which_file)])
var_list= [x for x in data]

In [ ]:


def PlotSounding(saveFig=True, hourUTC=None, utcOffset=-5,
                 titleSize=20, labelSize=18, tickSize=16, legendSize=16):
    # hourUTC: launch hour from the filename (assumed UTC). Defaults to the global `hour`.
    hourUTC = int(hour if hourUTC is None else hourUTC)
    hourLT = (hourUTC + utcOffset) % 24
    dateStr = datetime.strptime(date, '%Y%m%d').strftime('%d %B %Y')

    pressure    = data['pres'].values * units('hPa')
    temperature = data['tdry'].values * units('degC')
    dewpoint    = data['dp'].values   * units('degC')
    pressure, temperature, dewpoint, keep = clean_sounding(pressure, temperature, dewpoint)

    u_wind = data['u_wind'].values[keep] * units('m/s')
    v_wind = data['v_wind'].values[keep] * units('m/s')

    fig = plt.figure(figsize=(10, 10))
    skew = SkewT(fig, rotation=45)

    skew.plot(pressure, temperature, 'r', lw=2, label='Temperature')
    skew.plot(pressure, dewpoint, 'g', lw=2, label='Dewpoint')
    step = 40
    skew.plot_barbs(pressure[::step], u_wind[::step], v_wind[::step], y_clip_radius=0.01)

    lcl_p, lcl_T, lfc_p, lfc_T = calculate_lcl_lfc(pressure, temperature, dewpoint)
    skew.plot(lcl_p, lcl_T, 'ko', markerfacecolor='black', ms=10, label='LCL')
    skew.plot(lfc_p, lfc_T, 'bo', markerfacecolor='blue', ms=10, label='LFC')

    skew.plot_dry_adiabats()
    skew.plot_moist_adiabats()
    skew.plot_mixing_lines()

    ax = skew.ax
    # ax.set_title(f'{dateStr}, {hourLT:02d}00 LT ({hourUTC:02d}00 UTC)',
    #              fontsize=titleSize, pad=12)
    ax.set_xlabel('Temperature (\u00b0C)', fontsize=labelSize)
    ax.set_ylabel('Pressure (hPa)', fontsize=labelSize)
    ax.tick_params(axis='both', labelsize=tickSize)
    ax.legend(loc='upper left', fontsize=legendSize)

    if saveFig:
        os.makedirs("skewt", exist_ok=True)
        savePath=os.path.join("skewt", f"{date}.jpg")
        fig.savefig(savePath, dpi=200, bbox_inches='tight')
        print(f"Saved to {savePath}")
    return fig
        
def calculate_lcl_lfc(pressure, temperature, dewpoint):
    """Surface-based LCL and LFC. Returns (lcl_p, lcl_T, lfc_p, lfc_T)."""
    lcl_p, lcl_T = mpcalc.lcl(pressure[0], temperature[0], dewpoint[0])
    prof = mpcalc.parcel_profile(pressure, temperature[0], dewpoint[0]).to('degC')
    lfc_p, lfc_T = mpcalc.lfc(pressure, temperature, dewpoint, prof)
    return lcl_p, lcl_T.to('degC'), lfc_p, lfc_T.to('degC')

def clean_sounding(pressure, temperature, dewpoint):
    """Drop NaNs and enforce strictly decreasing pressure (needed by MetPy). Returns trimmed arrays and the mask."""
    p = pressure.magnitude; t = temperature.magnitude; td = dewpoint.magnitude
    ok = np.isfinite(p) & np.isfinite(t) & np.isfinite(td)

    keep = np.zeros(len(p), dtype=bool)
    pmin = np.inf
    for i in range(len(p)):
        if ok[i] and p[i] < pmin:
            keep[i] = True
            pmin = p[i]

    return (p[keep] * units('hPa'), t[keep] * units('degC'),
            td[keep] * units('degC'), keep)

fig = PlotSounding()

In [ ]:
def WindsTestPlot():
    
    plt.subplot(3,2,1)
    alt=data[var_list[int(var_list.index('alt'))]] 
    alt.plot()
    
    plt.subplot(3,2,2)
    tdry=data[var_list[int(var_list.index('tdry'))]] 
    pres=data[var_list[int(var_list.index('pres'))]] 
    tdry=tdry+273.15 #convert to Kelvin
    pot_temp=tdry*(1000/pres)**(0.2854) #convert to potential tmperature
    pot_temp.plot()
    pot_temp
    
    plt.subplot(3,2,3)
    RH=data[var_list[int(var_list.index('rh'))]]/100
    #Calculation for Mixing Ratio
    #Kerry Emanuel 1994
    Rd=287
    Rv=461
    eps= Rd/Rv
    e_star = np.exp(53.67957-(6743.769/tdry)-4.8451*np.log(tdry))
    e = RH*e_star
    # spec_humid = (eps*e)/((pres*100)-e*(1-eps))
    mix_ratio = (eps*e)*1000/(pres-e)
    mix_ratio.plot()
    
    #winds
    u_wind=data[var_list[int(var_list.index('u_wind'))]] 
    v_wind=data[var_list[int(var_list.index('v_wind'))]]
    print(f'original wind magnitude {u_wind.values**2+v_wind.values**2}')
    #rotate winds to match coastline orientation
    def rotation(theta):
        rotation_mat=np.array([[np.cos(theta), -np.sin(theta)],
                                [np.sin(theta), np.cos(theta)]])
        return rotation_mat
    
    #picks out each index of u and v 
    theta=-(45+90)
    rotation_mat=rotation(np.radians(theta))
    uv_vec=np.vstack([u_wind, v_wind]); uv_vec=np.dot(rotation_mat,uv_vec)
    u_wind=uv_vec[0];v_wind=uv_vec[1]
    print(f'rotated wind magnitude {u_wind**2+v_wind**2}')
    
    plt.subplot(3,2,4)
    plt.plot(u_wind)
    plt.subplot(3,2,5)
    plt.plot(v_wind)
    
    plt.tight_layout()

# WindsTestPlot()

In [ ]:
#MAKING SOUNDING FILE

def MakeSoundingFile():
    split=nc_filenames[int(which_file)].split('.')
    file_path = f'input_sounding_{split[2]}_{split[3]}.txt'
    
    # Open the file in write mode
    with open(file_path, 'w') as file:
        
        #this needs to be fixed to start at the correct value ***
        line = "{:>9.4f} {:>10.4f} {:>10.4f}\n".format(pres.values[0],pot_temp.values[0],mix_ratio.values[0])
        file.write(line)
            
        for i in range(0,len(alt)):
            var1=np.round(alt.values[i],4)
            var2=np.round(pot_temp.values[i],4)
            var3=np.round(mix_ratio.values[i],4)
            var4=np.round(u_wind[i],4);
            var5=np.round(v_wind[i],4);
    
            #zero velocity case #comment if not using
            # var4=0
            # var5=0
            
            line = "{:>9.4f} {:>10.4f} {:>10.4f} {:>10.4f} {:>10.4f}\n".format(var1, var2, var3, var4, var5)
            file.write(line)
MakeSoundingFile()